In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
import numpy as np

train = np.genfromtxt(
    "/kaggle/input/competitions/titanic/train.csv",
    delimiter=",",
    dtype=None,
    encoding="utf-8",
    skip_header=1
)

In [8]:
pclass = np.array([row[2] for row in train], dtype=np.float64)

sex = np.array([row[5] for row in train])         # FIXED
age = np.array([row[6] for row in train], dtype=np.float64)

sibsp = np.array([row[7] for row in train], dtype=np.float64)
parch = np.array([row[8] for row in train], dtype=np.float64)

fare = np.array([row[10] for row in train], dtype=np.float64)
embarked = np.array([row[12] for row in train])

y = np.array([row[1] for row in train], dtype=np.float64)

In [9]:
print(sex[:5])
print(age[:5])

['male' 'female' 'female' 'female' 'male']
[22. 38. 26. 35. 35.]


In [10]:
def encode_sex(sex):
    out = np.zeros(len(sex), dtype=np.float64)
    for i in range(len(sex)):
        if sex[i] == 'female':
            out[i] = 1.0
        else:
            out[i] = 0.0
    return out

sex = encode_sex(sex)

In [11]:
def encode_embarked(embarked):
    out = np.zeros(len(embarked), dtype=np.float64)
    for i in range(len(embarked)):
        if embarked[i] == 'C':
            out[i] = 0.0
        elif embarked[i] == 'Q':
            out[i] = 1.0
        else:   
            out[i] = 2.0
    return out

embarked = encode_embarked(embarked)

In [12]:
print(sex[:5])
print(embarked[:5])
print(sex.dtype, embarked.dtype)

[0. 1. 1. 1. 0.]
[2. 0. 2. 2. 2.]
float64 float64


In [13]:
# ensure float (important)
age = age.astype(np.float64)

# compute mean ignoring NaN
age_mean = np.nanmean(age)

# replace NaN with mean
for i in range(len(age)):
    if np.isnan(age[i]):
        age[i] = age_mean

In [14]:
fare = fare.astype(np.float64)

fare_mean = np.nanmean(fare)

for i in range(len(fare)):
    if np.isnan(fare[i]):
        fare[i] = fare_mean

In [15]:
print(np.isnan(age).sum())
print(np.isnan(fare).sum())

0
0


In [16]:
X = np.column_stack((
    pclass,
    sex,
    age,
    sibsp,
    parch,
    fare,
    embarked
)).astype(np.float64)

y = y.astype(np.float64)

In [17]:
print(X.shape)
print(X.dtype)

(891, 7)
float64


In [18]:
def standardize(X):
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)
    
    for i in range(len(std)):
        if std[i] == 0:
            std[i] = 1.0

    X_scaled = (X - mean) / std
    return X_scaled, mean, std

X, mean, std = standardize(X)

In [19]:
print(np.mean(X, axis=0))  
print(np.std(X, axis=0))   

[ 3.86272882e-17  2.93816598e-16  4.44768302e-15 -3.90509423e-16
 -2.00612690e-17  6.69062012e-16  3.94995173e-17]
[1. 1. 1. 1. 1. 1. 1.]


In [20]:
from numba import njit
import numpy as np

@njit
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

In [21]:
@njit
def train(X, y, lr=0.01, epochs=1000):
    n, d = X.shape
    
    w = np.zeros(d)
    b = 0.0
    
    for _ in range(epochs):
        z = X @ w + b
        y_hat = sigmoid(z)
        
        # gradients
        dw = (X.T @ (y_hat - y)) / n
        db = np.sum(y_hat - y) / n
        
        # update
        w -= lr * dw
        b -= lr * db
    
    return w, b

In [22]:
@njit
def predict(X, w, b):
    probs = sigmoid(X @ w + b)
    return (probs >= 0.5).astype(np.int32)

In [23]:
@njit
def accuracy(y, y_pred):
    return np.sum(y == y_pred) / len(y)

In [24]:
w, b = train(X, y, lr=0.01, epochs=1000)

In [25]:
y_pred = predict(X, w, b)
print("Accuracy:", accuracy(y, y_pred))

Accuracy: 0.792368125701459
